# Day28 Kaggle GPU Services

Run this notebook on Kaggle with GPU enabled. It avoids modifying Kaggle's base Python environment by installing dependencies into `/kaggle/working/day28_site` and adding that folder to `PYTHONPATH` only for the Day28 services.

It starts:

- OpenAI-compatible chat API on port `8001` using vLLM + `Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4` when available
- embedding API on port `8002`, using `BAAI/bge-small-en-v1.5` when available and deterministic fallback otherwise
- Cloudflare quick tunnels for both services

Copy the final `VLLM_NGROK_URL` and `EMBED_NGROK_URL` values into your local `.env`.

In [ ]:
# Cell 1 - install dependencies into an isolated target directory
import os
import shutil
import subprocess
import sys
from pathlib import Path

WORKDIR = Path('/kaggle/working')
TARGET_DIR = WORKDIR / 'day28_site'
BROKEN_VENV = WORKDIR / 'day28_venv'
PY = sys.executable

# Remove stale installs from earlier notebook versions or interrupted runs.
# The old venv can be missing pip, which causes `/bin/python: No module named pip`.
shutil.rmtree(BROKEN_VENV, ignore_errors=True)
shutil.rmtree(TARGET_DIR, ignore_errors=True)
TARGET_DIR.mkdir(parents=True, exist_ok=True)

def pip_install(*packages, check=True, target_dir=TARGET_DIR):
    cmd = [
        sys.executable, '-m', 'pip', 'install', '-q',
        '--disable-pip-version-check', '--no-warn-conflicts', '--upgrade',
        '--target', str(target_dir), *packages,
    ]
    print(' '.join(cmd))
    return subprocess.run(cmd, check=check, cwd=str(WORKDIR))

# Some Kaggle images can print `sitecustomize` errors if wrapt is missing from
# /kaggle/working after partial installs. Installing wrapt there is small and
# makes later Python startup quieter.
pip_install('wrapt==1.16.0', check=False, target_dir=WORKDIR)

pip_install('fastapi==0.115.6', 'uvicorn[standard]==0.32.1', 'requests==2.32.3', 'numpy==1.26.4')
# Lab guide uses sentence-transformers for embeddings. Install the package
# without pulling another dependency tree; Cell 5 falls back if deps/model fail.
pip_install('--no-deps', 'sentence-transformers==3.3.1', check=False)

# vLLM stack is installed inside the target package directory only. This follows the lab update,
# but uses --extra-index-url instead of --index-url so PyPI packages remain visible.
# If it fails, the notebook still starts a compatible fallback server so local
# Day28 can continue. Set INSTALL_VLLM=0 before this cell to skip the heavy install.
if os.environ.get('INSTALL_VLLM', '1') == '1':
    vllm_install = pip_install(
        'transformers', 'accelerate', 'vllm', 'xformers',
        '--extra-index-url', 'https://download.pytorch.org/whl/cu124',
        check=False,
    )
    print('vLLM install return code:', vllm_install.returncode)
    pip_install('grpcio>=1.60.0', check=False)
    # Do not install flashinfer-python on Kaggle T4. vLLM can load the 7B
    # model, but FlashInfer JIT often fails at runtime because Kaggle's image
    # does not expose libcuda.so for the linker. Force Triton/native paths.
    for flashinfer_path in list(TARGET_DIR.glob('flashinfer*')):
        if flashinfer_path.is_dir():
            shutil.rmtree(flashinfer_path, ignore_errors=True)
        else:
            flashinfer_path.unlink(missing_ok=True)
    shutil.rmtree(Path('/root/.cache/flashinfer'), ignore_errors=True)
    print('Skipped flashinfer-python install and removed stale FlashInfer files/cache')
    for ray_path in WORKDIR.glob('ray*'):
        if ray_path.is_dir():
            shutil.rmtree(ray_path, ignore_errors=True)
        else:
            ray_path.unlink(missing_ok=True)
    subprocess.run([sys.executable, '-m', 'pip', 'wheel', 'ray>=2.11', '-w', '/kaggle/working/packages'], check=False)
else:
    print('Skipping vLLM install because INSTALL_VLLM=0')

# Install cloudflared binary. This is free and does not require an account.
cloudflared = WORKDIR / 'cloudflared'
if not cloudflared.exists():
    subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '-O', str(cloudflared)], check=True)
    subprocess.run(['chmod', '+x', str(cloudflared)], check=True)

subprocess.run([str(cloudflared), '--version'], check=True)
print('Using Python:', PY)
print('Using dependency target:', TARGET_DIR)

In [ ]:
# Cell 2 - shared helpers and config
import os
import re
import socket
import subprocess
import sys
import textwrap
import threading
import time
from pathlib import Path

import requests

WORKDIR = Path('/kaggle/working')
PY = os.sys.executable
TARGET_DIR = WORKDIR / 'day28_site'
CLOUDFLARED = WORKDIR / 'cloudflared'
SERVICE_ENV = os.environ.copy()
SERVICE_ENV['PYTHONPATH'] = str(TARGET_DIR) + ':' + str(WORKDIR) + ':' + SERVICE_ENV.get('PYTHONPATH', '')
SERVICE_ENV['LD_LIBRARY_PATH'] = ':'.join([
    '/usr/local/nvidia/lib64',
    '/usr/local/cuda/compat',
    '/usr/local/cuda/lib64',
    '/usr/lib/x86_64-linux-gnu',
    SERVICE_ENV.get('LD_LIBRARY_PATH', ''),
])
SERVICE_ENV['VLLM_USE_FLASHINFER_SAMPLER'] = '0'
SERVICE_ENV['VLLM_ATTENTION_BACKEND'] = os.environ.get('VLLM_ATTENTION_BACKEND', 'TRITON_ATTN')
os.environ.update({
    'LD_LIBRARY_PATH': SERVICE_ENV['LD_LIBRARY_PATH'],
    'VLLM_USE_FLASHINFER_SAMPLER': SERVICE_ENV['VLLM_USE_FLASHINFER_SAMPLER'],
    'VLLM_ATTENTION_BACKEND': SERVICE_ENV['VLLM_ATTENTION_BACKEND'],
})
for path in [str(TARGET_DIR), str(WORKDIR)]:
    if path not in sys.path:
        sys.path.insert(0, path)

# Default follows the Day28 lab guide. If Kaggle cannot load this model, the
# notebook automatically falls back to an OpenAI-compatible local server.
MODEL_ID = os.environ.get('DAY28_MODEL_ID', 'Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4')
VLLM_PORT = int(os.environ.get('DAY28_VLLM_PORT', '8001'))
EMBED_PORT = int(os.environ.get('DAY28_EMBED_PORT', '8002'))

PROCESSES = []

def run_background(command, name, env=None):
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env or SERVICE_ENV, cwd=str(WORKDIR))
    PROCESSES.append((name, process))
    def stream_logs():
        assert process.stdout is not None
        for line in process.stdout:
            print(f'[{name}] {line}', end='')
    threading.Thread(target=stream_logs, name=f'{name}-logs', daemon=True).start()
    return process

def wait_for_http(url, timeout_seconds=300):
    deadline = time.time() + timeout_seconds
    last_error = None
    while time.time() < deadline:
        try:
            response = requests.get(url, timeout=5)
            if response.status_code < 500:
                return True
        except Exception as exc:
            last_error = exc
        time.sleep(5)
    print('Last wait error:', repr(last_error))
    return False

def wait_for_http_or_exit(url, process, timeout_seconds=300):
    deadline = time.time() + timeout_seconds
    last_error = None
    while time.time() < deadline:
        if process.poll() is not None:
            print(f'Process exited with code {process.returncode} before {url} became ready')
            return False
        try:
            response = requests.get(url, timeout=5)
            if response.status_code < 500:
                return True
        except Exception as exc:
            last_error = exc
        time.sleep(5)
    print('Last wait error:', repr(last_error))
    return False

def is_port_available(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        try:
            sock.bind(('127.0.0.1', port))
            return True
        except OSError:
            return False

def find_available_port(start_port: int) -> int:
    for port in range(start_port, start_port + 100):
        if is_port_available(port):
            return port
    raise RuntimeError(f'No available port found from {start_port}')

print('MODEL_ID:', MODEL_ID)
print('VLLM_PORT:', VLLM_PORT)
print('EMBED_PORT:', EMBED_PORT)

In [ ]:
# Cell 3 - write fallback chat server file
fallback_chat_server = f'''
import os
import time
import uuid
from fastapi import FastAPI
from pydantic import BaseModel

MODEL_ID = os.environ.get('MODEL_ID', '{MODEL_ID}')
app = FastAPI(title='Day28 OpenAI-compatible fallback chat server')

class ChatCompletionRequest(BaseModel):
    model: str | None = None
    messages: list[dict]

@app.get('/health')
def health():
    return {{'status': 'ok', 'mode': 'fallback-openai-compatible', 'model': MODEL_ID}}

@app.get('/v1/models')
def models():
    return {{'object': 'list', 'data': [{{'id': MODEL_ID, 'object': 'model', 'owned_by': 'day28'}}]}}

@app.post('/v1/chat/completions')
def chat(request: ChatCompletionRequest):
    user_messages = [m.get('content', '') for m in request.messages if m.get('role') == 'user']
    prompt = user_messages[-1] if user_messages else ''
    content = ('Kaggle fallback response: this endpoint is OpenAI-compatible and reachable. '
               'Use vLLM mode when the vLLM install and model load succeed. '
               f'Prompt preview: {{prompt[:500]}}')
    return {{
        'id': 'chatcmpl-' + uuid.uuid4().hex[:12],
        'object': 'chat.completion',
        'created': int(time.time()),
        'model': request.model or MODEL_ID,
        'choices': [{{'index': 0, 'message': {{'role': 'assistant', 'content': content}}, 'finish_reason': 'stop'}}],
        'usage': {{'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0}},
    }}
'''

(WORKDIR / 'fallback_chat_server.py').write_text(fallback_chat_server, encoding='utf-8')
print('Wrote fallback chat server to /kaggle/working')

In [ ]:
# Cell 4 - start chat server: vLLM first, fallback if vLLM is unavailable
shutil.rmtree(Path('/root/.cache/flashinfer'), ignore_errors=True)
print('VLLM_ATTENTION_BACKEND:', SERVICE_ENV.get('VLLM_ATTENTION_BACKEND'))
print('VLLM_USE_FLASHINFER_SAMPLER:', SERVICE_ENV.get('VLLM_USE_FLASHINFER_SAMPLER'))
check = subprocess.run([str(PY), '-c', 'import vllm; print(vllm.__version__)'], capture_output=True, text=True, env=SERVICE_ENV, cwd=str(WORKDIR))
print('vLLM import return code:', check.returncode)
print(check.stdout)
print(check.stderr[-1000:])

if check.returncode == 0:
    print('Starting vLLM OpenAI server...')
    chat_process = run_background([
        str(PY), '-m', 'vllm.entrypoints.openai.api_server',
        '--model', MODEL_ID,
        '--served-model-name', MODEL_ID,
        '--port', str(VLLM_PORT),
        '--host', '0.0.0.0',
        '--max-model-len', '2048',
        '--gpu-memory-utilization', '0.85',
        '--max-num-seqs', '1',
        '--enforce-eager',
        '--attention-backend', SERVICE_ENV.get('VLLM_ATTENTION_BACKEND', 'TRITON_ATTN'),
        '--trust-remote-code',
    ], 'vllm', env=SERVICE_ENV)
    ready = wait_for_http_or_exit(f'http://localhost:{VLLM_PORT}/v1/models', chat_process, timeout_seconds=900)
    if not ready:
        print('vLLM did not become ready. Terminating it before starting fallback server...')
        try:
            chat_process.terminate()
            chat_process.wait(timeout=30)
        except Exception as exc:
            print('Could not terminate vLLM cleanly:', repr(exc))
else:
    ready = False

if not ready:
    print('Starting fallback OpenAI-compatible chat server...')
    run_background([
        str(PY), '-m', 'uvicorn', 'fallback_chat_server:app',
        '--host', '0.0.0.0', '--port', str(VLLM_PORT), '--app-dir', str(WORKDIR)
    ], 'fallback-chat', env=SERVICE_ENV)
    assert wait_for_http(f'http://localhost:{VLLM_PORT}/v1/models', timeout_seconds=120)

print(requests.get(f'http://localhost:{VLLM_PORT}/v1/models', timeout=20).json())

In [ ]:
# Cell 5 - start embedding API
if wait_for_http(f'http://localhost:{EMBED_PORT}/health', timeout_seconds=5):
    print('Embedding API is already running on port', EMBED_PORT)
else:
    if not is_port_available(EMBED_PORT):
        old_port = EMBED_PORT
        EMBED_PORT = find_available_port(EMBED_PORT + 1)
        print(f'Port {old_port} is busy but not healthy; using port {EMBED_PORT} for embedding instead.')

    import hashlib
    import math
    from fastapi import FastAPI
    from pydantic import BaseModel
    import uvicorn

    VECTOR_SIZE = 384
    EMBED_MODEL_ID = os.environ.get('DAY28_EMBED_MODEL_ID', 'BAAI/bge-small-en-v1.5')
    EMBED_BACKEND = 'deterministic'
    EMBED_MODEL = None
    if os.environ.get('DAY28_USE_SENTENCE_TRANSFORMERS', '1') == '1':
        try:
            from sentence_transformers import SentenceTransformer
            print('Loading embedding model:', EMBED_MODEL_ID)
            EMBED_MODEL = SentenceTransformer(EMBED_MODEL_ID)
            EMBED_BACKEND = f'sentence-transformers:{EMBED_MODEL_ID}'
        except Exception as exc:
            print('SentenceTransformer unavailable; using deterministic embedding fallback:', repr(exc))

    embedding_app = FastAPI(title='Day28 deterministic embedding service')

    class EmbedRequest(BaseModel):
        texts: list[str]

    def deterministic_embedding(text: str, size: int = VECTOR_SIZE) -> list[float]:
        digest = hashlib.sha256(text.encode('utf-8')).digest()
        values = []
        while len(values) < size:
            for byte in digest:
                values.append((byte / 127.5) - 1.0)
                if len(values) == size:
                    break
            digest = hashlib.sha256(digest).digest()
        norm = math.sqrt(sum(v * v for v in values)) or 1.0
        return [v / norm for v in values]

    @embedding_app.get('/health')
    def embedding_health():
        return {'status': 'ok', 'mode': EMBED_BACKEND, 'vector_size': VECTOR_SIZE}

    @embedding_app.post('/embed')
    def embed(request: EmbedRequest):
        if EMBED_MODEL is not None:
            return {'embeddings': EMBED_MODEL.encode(request.texts).tolist()}
        return {'embeddings': [deterministic_embedding(text) for text in request.texts]}

    def run_embed():
        uvicorn.run(embedding_app, host='0.0.0.0', port=EMBED_PORT, log_level='info')

    EMBED_THREAD = threading.Thread(target=run_embed, name='embedding-api', daemon=True)
    EMBED_THREAD.start()
    assert wait_for_http(f'http://localhost:{EMBED_PORT}/health', timeout_seconds=120)

health = requests.get(f'http://localhost:{EMBED_PORT}/health', timeout=20).json()
sample = requests.post(f'http://localhost:{EMBED_PORT}/embed', json={'texts': ['AI platform integration']}, timeout=20).json()
print(health)
print('Embedding sample shape:', len(sample['embeddings']), len(sample['embeddings'][0]))

In [ ]:
# Cell 6 - expose both services with cloudflared
TUNNEL_PROCESSES = []

def expose_with_cloudflared(port: int) -> str:
    process = subprocess.Popen(
        [str(CLOUDFLARED), 'tunnel', '--url', f'http://localhost:{port}'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    TUNNEL_PROCESSES.append(process)
    assert process.stdout is not None
    for _ in range(180):
        line = process.stdout.readline()
        print(line, end='')
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            return match.group(0)
    raise RuntimeError(f'Could not find a trycloudflare URL for port {port}')

vllm_url = expose_with_cloudflared(VLLM_PORT)
embed_url = expose_with_cloudflared(EMBED_PORT)

print('\nCopy these into local .env:')
print(f'MODEL_ID={MODEL_ID}')
print(f'VLLM_NGROK_URL={vllm_url}')
print(f'EMBED_NGROK_URL={embed_url}')

In [ ]:
# Cell 7 - public endpoint checks
print('Chat models:')
print(requests.get(f'{vllm_url}/v1/models', timeout=30).json())

print('\nChat completion sample:')
chat = requests.post(
    f'{vllm_url}/v1/chat/completions',
    json={'model': MODEL_ID, 'messages': [{'role': 'user', 'content': 'Explain platform engineering in one sentence.'}]},
    timeout=60,
).json()
print(chat['choices'][0]['message']['content'])

print('\nEmbedding health:')
print(requests.get(f'{embed_url}/health', timeout=30).json())

print('\nEmbedding sample shape:')
sample = requests.post(f'{embed_url}/embed', json={'texts': ['AI platform integration']}, timeout=30).json()
print(len(sample['embeddings']), len(sample['embeddings'][0]))